In [1]:
"""
Test putbarablocktriplet — variable SDP 2×2, solution analytique exacte
=======================================================================

PROBLÈME :
  minimiser  tr(X)  =  X[0,0] + X[1,1]

  sous       X[0,0] + 2·X[1,0] + X[1,1]  =  4     (contrainte 0)
             X ⪰ 0,  X ∈ S²

MATRICE DE COEFFICIENT A (triangulaire inférieure) :
  (k=0, l=0) = 1   ← diagonal
  (k=1, l=0) = 1   ← hors-diagonal  →  MOSEK calcule 2·X[1,0]
  (k=1, l=1) = 1   ← diagonal

  La contrainte devient : X[0,0] + 2·X[1,0] + X[1,1] = 4
  ce qui s'écrit        : vᵀ X v = 4   avec v = [1, 1]ᵀ

SOLUTION ANALYTIQUE :
  X* = vvᵀ = [[1, 1],   (matrice de rang 1)
               [1, 1]]

  Vérif : vᵀ X* v = 1 + 2 + 1 = 4 ✓
  tr(X*) = 2  (valeur optimale)

  Borne inférieure : tr(X) ≥ vᵀXv / ‖v‖² = 4/2 = 2  → atteinte par X*.
"""

import numpy as np
import mosek

TOL = 1e-5

def solve():
    with mosek.Task() as task:

        # Variable SDP : 1 matrice 2×2
        task.appendbarvars([2])

        # 1 contrainte scalaire
        task.appendcons(1)

        # ── Objectif : tr(X) = <I, X> ──────────────────────────────────────
        # C = I₂  →  entrées diagonales uniquement
        task.putbarcblocktriplet(
            [0, 0],       # j : indice variable SDP
            [0, 1],       # k : ligne
            [0, 1],       # l : colonne  (k == l → diagonal)
            [1.0, 1.0],   # valeur
        )

        # ── Contrainte : <A, X> = 4 ────────────────────────────────────────
        # A[0,0]=1 (diag), A[1,0]=1 (hors-diag), A[1,1]=1 (diag)
        # → produit interne = X[0,0] + 2·X[1,0] + X[1,1]
        task.putbarablocktriplet(
            [0,   0,   0  ],   # i : contrainte
            [0,   0,   0  ],   # j : variable SDP
            [0,   1,   1  ],   # k : ligne   (k >= l)
            [0,   0,   1  ],   # l : colonne
            [1.0, 1.0, 1.0],   # valeur réelle de A[k,l]
        )

        # Borne : égalité à 4
        task.putconbound(0, mosek.boundkey.fx, 4.0, 4.0)

        task.optimize()

        barx = task.getbarxj(mosek.soltype.itr, 0)
        # barx = [X[0,0], X[1,0], X[1,1]]  (ordre triangulaire inférieur)
        X = np.array([
            [barx[0], barx[1]],
            [barx[1], barx[2]],
        ])
        return X


def check(X):
    X_ref   = np.array([[1., 1.], [1., 1.]])
    opt_ref = 2.0

    print("\n── Solution MOSEK ──────────────────────")
    print(np.array2string(X, precision=6, suppress_small=True))
    print(f"\n── Valeur optimale  tr(X*) = {np.trace(X):.6f}  (attendu {opt_ref})")

    tests = {
        "Valeur optimale (tr = 2)":
            abs(np.trace(X) - opt_ref) < TOL,
        "Contrainte (X[0,0]+2·X[1,0]+X[1,1] = 4)":
            abs(X[0,0] + 2*X[1,0] + X[1,1] - 4.0) < TOL,
        "SDP (λ_min ≥ 0)":
            np.linalg.eigvalsh(X).min() >= -TOL,
        "Proximité X* analytique":
            np.linalg.norm(X - X_ref, "fro") < 1e-3,
    }

    print()
    ok = True
    for label, passed in tests.items():
        print(f"  {'✅' if passed else '❌'}  {label}")
        ok = ok and passed

    print()
    print("✅ OK" if ok else "❌ ÉCHEC — vérifiez vos valeurs hors-diagonales")


if __name__ == "__main__":
    check(solve())


── Solution MOSEK ──────────────────────
[[1. 1.]
 [1. 1.]]

── Valeur optimale  tr(X*) = 2.000000  (attendu 2.0)

  ✅  Valeur optimale (tr = 2)
  ✅  Contrainte (X[0,0]+2·X[1,0]+X[1,1] = 4)
  ✅  SDP (λ_min ≥ 0)
  ✅  Proximité X* analytique

✅ OK


In [1]:
import pandas as pd

tab_jz = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_05_11_12h02_33s_test-jz-correction-mosek/results.csv")

In [2]:
tab_cnam = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_05_11_14h32_14s_comparaison_bornes_with_jz/results.csv")

In [3]:
tab_diff = abs(tab_cnam["optimal_value"] - tab_jz["optimal_value"])

In [7]:
tab_diff.max()

np.float64(6.205104980949727e-06)

In [5]:
(tab_cnam["data_index"] == tab_jz["data_index"]).sum()

np.int64(992)

In [6]:
for index in range(1000):
    if index not in tab_cnam["data_index"].values:
        print(f"Index {index} absent dans tab_cnam")
    elif index not in tab_jz["data_index"].values:
        print(f"Index {index} absent dans tab_jz")
    else : 
        index_tab_cnam = tab_cnam.index[tab_cnam["data_index"] == index][0]
        index_tab_jz = tab_jz.index[tab_jz["data_index"] == index][0]
        diff = abs(tab_cnam.loc[index_tab_cnam, "optimal_value"] - tab_jz.loc[index_tab_jz, "optimal_value"])
        if diff > 1e-3:
            print(f"Index {index} : différence de valeur optimale = {diff}")

Index 133 absent dans tab_cnam
Index 277 absent dans tab_cnam
Index 324 absent dans tab_cnam
Index 387 absent dans tab_cnam
Index 715 absent dans tab_cnam
Index 754 absent dans tab_cnam
Index 822 absent dans tab_cnam
Index 851 absent dans tab_cnam


In [3]:
import pandas as pd

In [4]:
tab_10 = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_05_18_13h47_04s_input-in-variables=10/part_0_200/results.csv")

In [5]:
tab_50 = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_05_18_13h49_07s_input-in-variables=50/part_0_200/results.csv")

In [6]:
tab_100 = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_05_18_13h51_01s_input-in-variables=100/part_0_200/results.csv")

In [10]:
(tab_50['optimal_value'] <= tab_100['optimal_value']).value_counts()

optimal_value
True     168
False     31
Name: count, dtype: int64

In [19]:
tab_50[~(tab_50['optimal_value'] <= tab_100['optimal_value'])][['status','optimal_value','primal_obj_value', 'dual_obj_value']]

,status,optimal_value,primal_obj_value,dual_obj_value
8,SolutionStatus.Optimal,3.555537,3.555537,3.555537
28,SolutionStatus.Optimal,3.553144,3.553144,3.553144
29,SolutionStatus.Optimal,3.401420,3.401420,3.401420
32,SolutionStatus.Optimal,3.361775,3.361775,3.361775
37,SolutionStatus.Optimal,3.493247,3.493247,3.493247
52,SolutionStatus.Optimal,-28.798044,-28.798044,-28.798044
60,SolutionStatus.Optimal,2.894366,2.894366,2.894366
61,SolutionStatus.Optimal,-2.211840,-2.211840,-2.211840
67,SolutionStatus.Optimal,3.436426,3.436426,3.436426
76,SolutionStatus.Optimal,3.802710,3.802710,3.802710


In [18]:
tab_100[~(tab_50['optimal_value'] <= tab_100['optimal_value'])][['status','optimal_value','primal_obj_value', 'dual_obj_value']]

,status,optimal_value,primal_obj_value,dual_obj_value
8,SolutionStatus.Optimal,3.507227,3.507227,3.507227
28,SolutionStatus.Optimal,3.498721,3.498721,3.498721
29,SolutionStatus.Optimal,3.387691,3.387691,3.387691
32,SolutionStatus.Optimal,3.339541,3.339541,3.339541
37,SolutionStatus.Optimal,3.365953,3.365953,3.365953
52,SolutionStatus.Optimal,-28.798046,-28.798046,-28.798046
60,SolutionStatus.Optimal,2.867871,2.867871,2.867871
61,SolutionStatus.Optimal,-2.211840,-2.211840,-2.211840
67,SolutionStatus.Optimal,3.329799,3.329799,3.329799
76,SolutionStatus.Optimal,3.726866,3.726866,3.726866


In [ ]:
import pandas as pd

In [ ]:
tab_mosek = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_06_09_18h37_16s_LANCE_AVEC_MOSEK/results.csv")

In [ ]:
tab_cvxpy = pd.read_csv("/share/homes/boyerma/FastSDPCertification/results/benchmark/blob_nn_4x10-0.5/2026_06_09_18h35_57s_LANCE_AVEC_CVXPY/results.csv")

In [ ]:
tab_mosek.head(10)['optimal_value']

In [ ]:
tab_cvxpy.head(10)['optimal_value']